# 🧹 Notebook 03 – Data Cleaning and Preprocessing
**Goal:** turn messy tweets into clean text a model can learn from, and save the result to `01_Data/Processed/cleaned_tweets.csv`.

**Important design decisions**
- Only `text` (input) and `airline_sentiment` (target) are kept for modelling → **no leakage**.
- **Negation words** (`not`, `no`, `never`, …) are **kept**, because *"not good"* means the opposite of *"good"*.
- `@airline` mentions are removed so the model learns language, not airline names.
- Rows that become empty after cleaning are removed.

## 1. Import Libraries and Load Raw Data

In [ ]:
# Import the libraries used for cleaning
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)

In [ ]:
# Set up project paths (works on your D: drive path, or falls back to the parent of 02_Notebooks)
from pathlib import Path

BASE_DIR = Path(r"D:\DataScience\notebooks\SentimentIQ")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent          # fallback: notebook is running inside 02_Notebooks/

RAW_PATH       = BASE_DIR / "01_Data" / "Raw" / "Tweets.csv"
PROCESSED_DIR  = BASE_DIR / "01_Data" / "Processed"
CLEANED_PATH   = PROCESSED_DIR / "cleaned_tweets.csv"
MODELS_DIR     = BASE_DIR / "03_Models"
REPORTS_DIR    = BASE_DIR / "05_Reports"

for folder in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42      # same seed everywhere -> reproducible results
TEST_SIZE    = 0.20    # 80% training / 20% testing

print("Project folder :", BASE_DIR)
print("Raw data exists:", RAW_PATH.exists())

In [ ]:
# Load the raw dataset and start a log that records how many rows remain after every step
try:
    raw = pd.read_csv(RAW_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Tweets.csv not found at {RAW_PATH}")

df = raw.copy()                       # work on a copy; the raw data stays untouched
log = [("Original dataset", len(df))]
print("Original shape:", df.shape)

## 2. Remove Duplicate Records

In [ ]:
# Remove fully identical rows, then repeated tweet text (same text can leak from train into test)
conflicts = (df.groupby("text")["airline_sentiment"].nunique() > 1).sum()
print("Texts that appear with different labels:", conflicts)

before = len(df)
df = df.drop_duplicates()
print("Identical rows removed :", before - len(df))

before = len(df)
df = df.drop_duplicates(subset="text", keep="first")
print("Repeated text removed  :", before - len(df))

log.append(("After removing duplicates", len(df)))

## 3. Handle Missing Text Values

In [ ]:
# Drop rows where the text or the sentiment label is missing / blank
before = len(df)
df = df.dropna(subset=["text", "airline_sentiment"])
df = df[df["text"].astype(str).str.strip() != ""]
print("Rows removed:", before - len(df))
log.append(("After removing missing text/labels", len(df)))

## 4. Check Invalid Sentiment Labels

In [ ]:
# Only 'positive', 'negative' and 'neutral' are valid; remove everything else
VALID_LABELS = {"positive", "negative", "neutral"}
df["airline_sentiment"] = df["airline_sentiment"].astype(str).str.strip().str.lower()

invalid = ~df["airline_sentiment"].isin(VALID_LABELS)
print("Invalid labels found:", invalid.sum())
df = df[~invalid]
print("Remaining labels:", sorted(df["airline_sentiment"].unique()))
log.append(("After removing invalid labels", len(df)))

## 5. Remove Unnecessary Columns for Model Training

In [ ]:
# Keep ONLY the input (text) and the target (airline_sentiment)
dropped = [col for col in df.columns if col not in ("text", "airline_sentiment")]
model_df = df[["text", "airline_sentiment"]].copy()

print("Columns kept   :", list(model_df.columns))
print("Columns dropped:", dropped)

| Dropped column | Reason |
|---|---|
| `tweet_id`, `name` | identifiers – no linguistic meaning, risk of memorising users |
| `airline_sentiment_confidence`, `*_gold` | derived from the label → **leakage** |
| `negativereason(_confidence)` | exists only for negative tweets → **leakage** |
| `airline` | would let the model guess sentiment from the airline instead of the language |
| `tweet_created`, `tweet_location`, `user_timezone`, `tweet_coord`, `retweet_count` | not text; mostly missing or irrelevant |

## 6. Build the Text-Cleaning Toolkit
This cell contains the complete preprocessing logic. In Notebook 08 the very same code is saved as `04_App/preprocessing.py`, so the **web app cleans text exactly like the training data**.

> First run only: NLTK downloads `stopwords` and `wordnet` (internet needed once).

In [ ]:
# preprocessing.py - text cleaning shared by the notebooks and the Streamlit app
import html
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer

# Download the NLTK resources once (quietly skipped if already installed / offline)
for _resource in ("stopwords", "wordnet", "omw-1.4"):
    try:
        nltk.download(_resource, quiet=True)
    except Exception:
        pass

# Words that can flip the meaning of a sentence must NOT be removed
NEGATION_WORDS = {"no", "nor", "not", "never", "neither", "none",
                  "nothing", "nobody", "nowhere", "cannot", "without"}
KEEP_WORDS = NEGATION_WORDS | {"very", "too", "but"}   # negations + a few strong modifiers

STOP_WORDS = set(stopwords.words("english")) - KEEP_WORDS

# Expand contractions so that "don't" becomes "do not" (keeps the negation visible)
CONTRACTIONS = [
    (r"won't", "will not"), (r"can't", "can not"), (r"cannot", "can not"),
    (r"shan't", "shall not"), (r"n't", " not"), (r"'re", " are"),
    (r"'ve", " have"), (r"'ll", " will"), (r"'d", " would"), (r"'m", " am"),
]

_tokenizer = RegexpTokenizer(r"[a-z]+")
_lemmatizer = WordNetLemmatizer()


def normalize_text(text):
    # Lowercase, strip HTML/URLs/mentions/hashtag symbols, expand contractions, keep letters only
    text = html.unescape(str(text)).lower()
    text = text.replace("\u2019", "'").replace("\u2018", "'")      # curly apostrophes
    text = re.sub(r"</?[a-z][^>]*>", " ", text)                    # HTML tags
    text = re.sub(r"(https?://\S+|www\.\S+)", " ", text)           # URLs
    text = re.sub(r"@\w+", " ", text)                              # @mentions (airline handles / usernames)
    text = text.replace("#", " ")                                  # keep hashtag word, drop the symbol
    for pattern, replacement in CONTRACTIONS:
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)                     # sooooo -> soo
    text = re.sub(r"[^a-z\s]", " ", text)                          # punctuation, digits, emojis
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text):
    return _tokenizer.tokenize(text)


def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 1]


def lemmatize_tokens(tokens):
    lemmas = []
    for token in tokens:
        lemma = _lemmatizer.lemmatize(token, pos="v")   # delayed -> delay
        lemma = _lemmatizer.lemmatize(lemma, pos="n")   # flights -> flight
        lemmas.append(lemma)
    return lemmas


def clean_text(text):
    # Full pipeline. Returns "" when the input is empty or has no useful words left.
    if not isinstance(text, str) or not text.strip():
        return ""
    tokens = tokenize(normalize_text(text))
    tokens = remove_stopwords(tokens)
    tokens = lemmatize_tokens(tokens)
    return " ".join(tokens)

In [ ]:
# Show which stop words are kept on purpose (negations can reverse sentiment)
print("Total stop words used :", len(STOP_WORDS))
print("Negations preserved   :", sorted(NEGATION_WORDS))
print("'not' removed?        :", "not" in STOP_WORDS, "| 'no' removed?:", "no" in STOP_WORDS)

## 7. Step-by-Step Demonstration on One Tweet

In [ ]:
# A sample tweet containing a mention, HTML, URL, hashtag, emoji, contraction and a negation
import re, html
sample = "@united I DON'T like the delay!!! <b>Terrible</b> service :( https://t.co/abc123 #NeverAgain &amp; 3 hrs late 😡"
print("ORIGINAL :", sample)

### 7.1 Convert text to lowercase

In [ ]:
# Lowercase so "Delay" and "delay" are treated as the same word
step = html.unescape(sample).lower()
print(step)

### 7.2 Remove URLs

In [ ]:
# Remove web links (they carry no sentiment)
step = re.sub(r"(https?://\S+|www\.\S+)", " ", step)
print(step)

### 7.3 Remove HTML tags

In [ ]:
# Remove HTML tags such as <b> ... </b>
step = re.sub(r"</?[a-z][^>]*>", " ", step)
print(step)

### 7.4 Remove unnecessary special characters

In [ ]:
# Remove @mentions and the '#' symbol (the hashtag word itself is kept)
step = re.sub(r"@\w+", " ", step)
step = step.replace("#", " ")
print(step)

### 7.5 Handle punctuation (contractions first!)

In [ ]:
# Expand contractions BEFORE removing punctuation, otherwise "don't" would become "don t" and lose its negation
for pattern, replacement in CONTRACTIONS:
    step = re.sub(pattern, replacement, step)
step = re.sub(r"(.)\1{2,}", r"\1\1", step)        # "sooooo" -> "soo"
step = re.sub(r"[^a-z\s]", " ", step)             # remove punctuation, digits, emojis
step = re.sub(r"\s+", " ", step).strip()
print(step)

### 7.6 Tokenization

In [ ]:
# Split the sentence into individual words (tokens)
tokens = tokenize(step)
print(tokens)

### 7.7 Stopword removal (negations preserved)

In [ ]:
# Remove very common words (the, I, a ...) but keep 'not', 'no', 'never'
tokens = remove_stopwords(tokens)
print(tokens)

### 7.8 Lemmatization

In [ ]:
# Reduce words to their dictionary form: "delayed" -> "delay", "flights" -> "flight"
tokens = lemmatize_tokens(tokens)
print(tokens)
print("\nFINAL (via clean_text):", clean_text(sample))

**Expected output (approximately):** `not like delay terrible service neveragain hrs late` – the negation *not* survives, while the URL, mention, HTML, emoji and filler words are gone.

## 8. Create the `cleaned_text` Column

In [ ]:
# Apply the full pipeline to every tweet (takes a few seconds)
model_df = model_df.copy()
model_df["cleaned_text"] = model_df["text"].apply(clean_text)
log.append(("After text cleaning (before empty check)", len(model_df)))
model_df.head()

## 9. Handle Empty Text After Preprocessing

In [ ]:
# Some tweets contain only a mention/URL/emoji -> nothing is left after cleaning
empty_mask = model_df["cleaned_text"].str.strip() == ""
print("Tweets that became empty:", empty_mask.sum())
display(model_df.loc[empty_mask, ["text", "cleaned_text"]].head())

model_df = model_df[~empty_mask]
log.append(("After removing empty cleaned text", len(model_df)))

## 10. Remove Near-Duplicates After Cleaning

In [ ]:
# Different raw tweets (e.g. different @mentions) can become identical after cleaning -> keep one copy
before = len(model_df)
model_df = model_df.drop_duplicates(subset="cleaned_text", keep="first")
print("Near-duplicates removed:", before - len(model_df))
log.append(("After removing cleaned-text duplicates", len(model_df)))

## 11. Verify That Negations Were Preserved

In [ ]:
# Check that negation words still exist in the cleaned text
pattern = r"\b(?:not|no|never|cannot|nothing|without)\b"
has_neg_original = model_df["text"].str.lower().str.contains(r"\b(?:not|no|never)\b|n't", regex=True)
has_neg_cleaned  = model_df["cleaned_text"].str.contains(pattern, regex=True)

print(f"Tweets with a negation in the original : {has_neg_original.sum():,}")
print(f"Tweets with a negation after cleaning  : {has_neg_cleaned.sum():,}")
display(model_df[has_neg_cleaned][["text", "cleaned_text"]].head(5))

## 12. Compare Original Text and Cleaned Text

In [ ]:
# Look at 10 random before/after examples
model_df.sample(10, random_state=RANDOM_STATE)[["airline_sentiment", "text", "cleaned_text"]]

In [ ]:
# Compare average length before and after cleaning
comparison = pd.DataFrame({
    "avg_characters": [model_df["text"].str.len().mean(), model_df["cleaned_text"].str.len().mean()],
    "avg_words":      [model_df["text"].str.split().str.len().mean(), model_df["cleaned_text"].str.split().str.len().mean()],
}, index=["Original text", "Cleaned text"]).round(1)
comparison

## 13. Final Check and Save the Processed Dataset

In [ ]:
# Final dataset: reset the index, check for problems and look at class balance
model_df = model_df.reset_index(drop=True)

print("Final shape       :", model_df.shape)
print("Missing values    :", model_df.isnull().sum().sum())
print("Empty cleaned_text:", (model_df["cleaned_text"].str.strip() == "").sum())
print()
print(model_df["airline_sentiment"].value_counts())

In [ ]:
# Save to 01_Data/Processed/cleaned_tweets.csv
try:
    model_df.to_csv(CLEANED_PATH, index=False, encoding="utf-8")
    print("✅ Saved:", CLEANED_PATH)
except PermissionError:
    print("❌ Cannot write the file - close it in Excel and run this cell again.")

In [ ]:
# Reload the saved file to confirm it was written correctly + show the cleaning summary
check = pd.read_csv(CLEANED_PATH)
assert check.shape == model_df.shape, "Saved file does not match the DataFrame!"
print("Reloaded shape:", check.shape)

log.append(("FINAL dataset", len(check)))
pd.DataFrame(log, columns=["Step", "Rows remaining"])

## ✅ Summary
- Duplicates, missing values, invalid labels and empty texts were removed.
- Only `text`, `airline_sentiment` and the new `cleaned_text` were kept.
- Negations were preserved, URLs / HTML / mentions / emojis / punctuation were removed, and words were lemmatized.
- Output file: `01_Data/Processed/cleaned_tweets.csv`.

**Next:** Notebook 04 – Text Feature Extraction (TF-IDF).